# Hybrid Recommender System: Collaborative Filtering + Content-Based

## Overview
This notebook implements a hybrid recommender system that combines:
- **Collaborative Filtering** (CF): User-Based and Item-Based approaches
- **Content-Based Filtering**: Using movie genre information

## Goals
1. Address cold-start problem by leveraging content features
2. Improve prediction accuracy for users/movies with limited interaction history
3. Demonstrate weighted hybrid approach with adaptive weighting

## Hybrid Strategy
- **Warm-start** (user & movie in training): `score = 0.7 × CF + 0.3 × content`
- **Cold-start user**: `score = 0.3 × CF + 0.7 × content`
- **Cold-start movie**: `score = 0.3 × CF + 0.7 × content`
- **Double cold-start**: `score = 1.0 × content` (popularity-based)

In [ ]:
import pandas as pd
import numpy as np
import ast
from datetime import datetime
import time
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import csr_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Libraries imported successfully")
print(f"Timestamp: {datetime.now()}")

## 1. Load Data

In [ ]:
# Load train/test splits (temporal split for realistic evaluation)
print("Loading temporal split datasets...")
train = pd.read_csv('../../datasets/output/split_and_train_datasets/temporal_split/train_ratings.csv')
test = pd.read_csv('../../datasets/output/split_and_train_datasets/temporal_split/test_ratings.csv')

# Load movie metadata with genres
print("Loading movie metadata...")
movies = pd.read_csv('../../datasets/output/cleaned_datasets/cleaned_movies_metadata.csv')
links = pd.read_csv('../../datasets/output/cleaned_datasets/cleaned_links.csv')

print(f"\nTrain: {len(train):,} ratings")
print(f"Test: {len(test):,} ratings")
print(f"Movies: {len(movies):,} movies")
print(f"Links: {len(links):,} links")

## 2. Prepare Movie Genre Features

In [ ]:
# Parse genre strings to lists
def parse_genres(genre_str):
    """Convert string representation of list to actual list"""
    if pd.isna(genre_str) or genre_str == '[]':
        return []
    try:
        return ast.literal_eval(genre_str)
    except:
        return []

movies['genres'] = movies['genres_list'].apply(parse_genres)

# Merge with links to get movieId
movies_with_ids = movies.merge(links[['movieId', 'tmdbId']], 
                                left_on='id', right_on='tmdbId', how='inner')

print(f"Movies with genres: {len(movies_with_ids):,}")
print(f"\nSample genres:")
print(movies_with_ids[['movieId', 'title', 'genres']].head())

# Get all unique genres
all_genres = set()
for genres in movies_with_ids['genres']:
    all_genres.update(genres)

all_genres = sorted(list(all_genres))
print(f"\nTotal unique genres: {len(all_genres)}")
print(f"Genres: {all_genres}")

In [ ]:
# Create genre one-hot encoding for each movie
genre_matrix = []
movie_ids = []

for _, row in movies_with_ids.iterrows():
    movie_ids.append(row['movieId'])
    genre_vector = [1 if genre in row['genres'] else 0 for genre in all_genres]
    genre_matrix.append(genre_vector)

genre_df = pd.DataFrame(genre_matrix, columns=all_genres, index=movie_ids)

print(f"Genre feature matrix: {genre_df.shape}")
print(f"\nSample genre features:")
print(genre_df.head())

## 3. Build Collaborative Filtering Components

In [ ]:
# Create user-item matrix from training data
print("Building user-item matrix...")
train_matrix = train.pivot_table(index='userId', columns='movieId', values='rating')

print(f"User-item matrix: {train_matrix.shape}")
print(f"Sparsity: {(1 - train_matrix.count().sum() / (train_matrix.shape[0] * train_matrix.shape[1])) * 100:.2f}%")

# Get sets of known users and movies
train_users = set(train['userId'].unique())
train_movies = set(train['movieId'].unique())

print(f"\nTrain users: {len(train_users):,}")
print(f"Train movies: {len(train_movies):,}")

In [ ]:
# Compute item-item similarity (for item-based CF)
print("Computing item-item similarity matrix...")
start_time = time.time()

# Fill NaN with 0 for similarity computation
train_matrix_filled = train_matrix.fillna(0)

# Compute cosine similarity between items (movies)
item_similarity = cosine_similarity(train_matrix_filled.T)
item_sim_df = pd.DataFrame(item_similarity,
                           index=train_matrix.columns,
                           columns=train_matrix.columns)

elapsed = time.time() - start_time
print(f"Item similarity matrix computed in {elapsed:.2f} seconds")
print(f"Shape: {item_sim_df.shape}")

## 4. Implement Content-Based Scoring

In [ ]:
def get_user_genre_profile(user_id, train_data, genre_df):
    """
    Build a user's genre preference profile based on their rating history.
    Returns a weighted genre vector where higher values indicate preference.
    """
    # Get user's ratings
    user_ratings = train_data[train_data['userId'] == user_id]
    
    if len(user_ratings) == 0:
        # Return uniform distribution for cold-start users
        return pd.Series(0, index=genre_df.columns)
    
    # Initialize genre preference vector
    genre_profile = pd.Series(0.0, index=genre_df.columns)
    
    # Accumulate weighted genre preferences
    for _, row in user_ratings.iterrows():
        movie_id = row['movieId']
        rating = row['rating']
        
        if movie_id in genre_df.index:
            # Weight by rating (higher ratings = stronger preference)
            genre_profile += genre_df.loc[movie_id] * rating
    
    # Normalize by total ratings
    if genre_profile.sum() > 0:
        genre_profile = genre_profile / genre_profile.sum()
    
    return genre_profile

# Test with a sample user
sample_user = train['userId'].iloc[0]
sample_profile = get_user_genre_profile(sample_user, train, genre_df)

print(f"Genre profile for user {sample_user}:")
print(sample_profile[sample_profile > 0].sort_values(ascending=False).head(10))

In [ ]:
def content_based_score(user_id, movie_id, train_data, genre_df, global_mean):
    """
    Compute content-based score for a user-movie pair using genre similarity.
    """
    # Get user's genre profile
    user_profile = get_user_genre_profile(user_id, train_data, genre_df)
    
    if movie_id not in genre_df.index:
        # Movie not in genre database - use global mean
        return global_mean
    
    # Get movie's genre vector
    movie_genres = genre_df.loc[movie_id]
    
    # Compute similarity (dot product of normalized vectors)
    if user_profile.sum() == 0 or movie_genres.sum() == 0:
        # No genre overlap or no user history
        return global_mean
    
    # Cosine similarity
    similarity = np.dot(user_profile, movie_genres)
    
    # Scale to rating range [0.5, 5.0]
    # Map similarity [0, 1] to rating range, centered around global mean
    score = global_mean + (similarity - 0.5) * 2.0
    
    # Clip to valid range
    score = np.clip(score, 0.5, 5.0)
    
    return score

# Test content-based scoring
global_mean = train['rating'].mean()
sample_movie = train['movieId'].iloc[0]
cb_score = content_based_score(sample_user, sample_movie, train, genre_df, global_mean)

print(f"\nContent-based score for user {sample_user}, movie {sample_movie}: {cb_score:.3f}")
print(f"Global mean: {global_mean:.3f}")

## 5. Implement Collaborative Filtering Prediction

In [ ]:
def item_based_cf_predict(user_id, movie_id, train_matrix, item_sim_df, k=30, global_mean=3.5):
    """
    Item-based collaborative filtering prediction.
    """
    # Check if movie and user are in training data
    if movie_id not in item_sim_df.columns:
        return global_mean, False  # Return (prediction, is_cf_prediction)
    
    if user_id not in train_matrix.index:
        return global_mean, False
    
    # Get user's ratings
    user_ratings = train_matrix.loc[user_id]
    
    # Find similar items that the user has rated
    similar_items = item_sim_df[movie_id].sort_values(ascending=False)[1:k+1]
    
    # Compute weighted average
    numerator = 0.0
    denominator = 0.0
    
    for item, sim in similar_items.items():
        if pd.notna(user_ratings[item]):  # User rated this item
            numerator += sim * user_ratings[item]
            denominator += abs(sim)
    
    if denominator == 0:
        return global_mean, False
    
    prediction = numerator / denominator
    return np.clip(prediction, 0.5, 5.0), True

# Test CF prediction
cf_pred, is_cf = item_based_cf_predict(sample_user, sample_movie, train_matrix, item_sim_df, global_mean=global_mean)
print(f"CF prediction for user {sample_user}, movie {sample_movie}: {cf_pred:.3f} (CF: {is_cf})")

## 6. Implement Hybrid Prediction with Adaptive Weighting

In [ ]:
def hybrid_predict(user_id, movie_id, train_data, train_matrix, item_sim_df, 
                   genre_df, train_users, train_movies, global_mean, k=30):
    """
    Hybrid prediction combining CF and content-based filtering with adaptive weighting.
    
    Weighting strategy:
    - Warm-start (both in training): CF=0.7, Content=0.3
    - Cold-start user: CF=0.3, Content=0.7
    - Cold-start movie: CF=0.3, Content=0.7
    - Double cold-start: CF=0.0, Content=1.0
    """
    # Get CF prediction
    cf_score, is_cf = item_based_cf_predict(user_id, movie_id, train_matrix, 
                                            item_sim_df, k=k, global_mean=global_mean)
    
    # Get content-based prediction
    cb_score = content_based_score(user_id, movie_id, train_data, genre_df, global_mean)
    
    # Determine weighting based on cold-start status
    user_is_warm = user_id in train_users
    movie_is_warm = movie_id in train_movies
    
    if user_is_warm and movie_is_warm:
        # Warm-start: trust CF more
        cf_weight = 0.7
        cb_weight = 0.3
        case = 'warm'
    elif user_is_warm or movie_is_warm:
        # Partial cold-start: balance CF and content
        cf_weight = 0.3
        cb_weight = 0.7
        case = 'partial_cold'
    else:
        # Double cold-start: use content only
        cf_weight = 0.0
        cb_weight = 1.0
        case = 'double_cold'
    
    # Compute hybrid score
    hybrid_score = cf_weight * cf_score + cb_weight * cb_score
    
    return np.clip(hybrid_score, 0.5, 5.0), case

# Test hybrid prediction on different scenarios
print("Testing hybrid prediction on different scenarios:\n")

# Warm-start case
warm_user = train['userId'].iloc[100]
warm_movie = train['movieId'].iloc[100]
pred, case = hybrid_predict(warm_user, warm_movie, train, train_matrix, item_sim_df,
                            genre_df, train_users, train_movies, global_mean)
print(f"Warm-start: user {warm_user}, movie {warm_movie}")
print(f"  Prediction: {pred:.3f}, Case: {case}")

# Get actual cold-start examples from test set
test_users = set(test['userId'].unique())
test_movies = set(test['movieId'].unique())

cold_start_users = test_users - train_users
cold_start_movies = test_movies - train_movies

print(f"\nCold-start statistics:")
print(f"  Cold-start users: {len(cold_start_users):,} / {len(test_users):,} ({len(cold_start_users)/len(test_users)*100:.1f}%)")
print(f"  Cold-start movies: {len(cold_start_movies):,} / {len(test_movies):,} ({len(cold_start_movies)/len(test_movies)*100:.1f}%)")

## 7. Evaluate Hybrid System on Test Set

In [ ]:
# Sample test set for faster evaluation (use full test set for final results)
# For development: use 10,000 samples
# For final: use all test samples

USE_SAMPLE = True  # Set to False for final evaluation
SAMPLE_SIZE = 10000

if USE_SAMPLE:
    test_sample = test.sample(n=min(SAMPLE_SIZE, len(test)), random_state=42)
    print(f"Using sample of {len(test_sample):,} test ratings for faster evaluation")
else:
    test_sample = test
    print(f"Using full test set of {len(test_sample):,} ratings")

print(f"\nStarting hybrid evaluation...")
start_time = time.time()

predictions = []
actuals = []
cases = []

for idx, row in test_sample.iterrows():
    if idx % 1000 == 0:
        elapsed = time.time() - start_time
        rate = idx / elapsed if elapsed > 0 else 0
        print(f"  Processed {idx:,} / {len(test_sample):,} ({idx/len(test_sample)*100:.1f}%) - {rate:.1f} ratings/sec")
    
    user_id = row['userId']
    movie_id = row['movieId']
    actual = row['rating']
    
    pred, case = hybrid_predict(user_id, movie_id, train, train_matrix, item_sim_df,
                                genre_df, train_users, train_movies, global_mean)
    
    predictions.append(pred)
    actuals.append(actual)
    cases.append(case)

elapsed = time.time() - start_time
print(f"\nEvaluation completed in {elapsed:.2f} seconds")
print(f"Average speed: {len(test_sample)/elapsed:.1f} ratings/sec")

In [ ]:
# Compute metrics
rmse = np.sqrt(mean_squared_error(actuals, predictions))
mae = mean_absolute_error(actuals, predictions)

print(f"\n{'='*80}")
print(f"HYBRID SYSTEM EVALUATION RESULTS")
print(f"{'='*80}")
print(f"\nOverall Metrics:")
print(f"  RMSE: {rmse:.6f}")
print(f"  MAE:  {mae:.6f}")
print(f"  Test samples: {len(test_sample):,}")

# Break down by case type
results_df = pd.DataFrame({
    'actual': actuals,
    'predicted': predictions,
    'case': cases
})

print(f"\n{'='*80}")
print(f"BREAKDOWN BY CASE TYPE")
print(f"{'='*80}")

for case_type in ['warm', 'partial_cold', 'double_cold']:
    case_df = results_df[results_df['case'] == case_type]
    if len(case_df) > 0:
        case_rmse = np.sqrt(mean_squared_error(case_df['actual'], case_df['predicted']))
        case_mae = mean_absolute_error(case_df['actual'], case_df['predicted'])
        
        print(f"\n{case_type.upper()}:")
        print(f"  Count: {len(case_df):,} ({len(case_df)/len(results_df)*100:.2f}%)")
        print(f"  RMSE:  {case_rmse:.6f}")
        print(f"  MAE:   {case_mae:.6f}")

## 8. Compare with Pure CF Baseline

In [ ]:
# Run pure CF for comparison
print(f"Running pure Item-Based CF baseline for comparison...\n")
start_time = time.time()

cf_predictions = []
cf_fallback_count = 0

for idx, row in test_sample.iterrows():
    if idx % 1000 == 0:
        elapsed = time.time() - start_time
        rate = idx / elapsed if elapsed > 0 else 0
        print(f"  Processed {idx:,} / {len(test_sample):,} ({idx/len(test_sample)*100:.1f}%) - {rate:.1f} ratings/sec")
    
    user_id = row['userId']
    movie_id = row['movieId']
    
    pred, is_cf = item_based_cf_predict(user_id, movie_id, train_matrix, item_sim_df, 
                                        k=30, global_mean=global_mean)
    
    cf_predictions.append(pred)
    if not is_cf:
        cf_fallback_count += 1

elapsed = time.time() - start_time
print(f"\nCF baseline completed in {elapsed:.2f} seconds")

# Compute CF metrics
cf_rmse = np.sqrt(mean_squared_error(actuals, cf_predictions))
cf_mae = mean_absolute_error(actuals, cf_predictions)

print(f"\n{'='*80}")
print(f"PURE CF BASELINE RESULTS")
print(f"{'='*80}")
print(f"\nMetrics:")
print(f"  RMSE: {cf_rmse:.6f}")
print(f"  MAE:  {cf_mae:.6f}")
print(f"  Fallback to global mean: {cf_fallback_count:,} ({cf_fallback_count/len(test_sample)*100:.2f}%)")

In [ ]:
# Comparison summary
print(f"\n{'='*80}")
print(f"HYBRID vs PURE CF COMPARISON")
print(f"{'='*80}")
print(f"\n{'Algorithm':<25} {'RMSE':<12} {'MAE':<12} {'Improvement'}")
print(f"{'-'*25} {'-'*12} {'-'*12} {'-'*20}")
print(f"{'Pure Item-Based CF':<25} {cf_rmse:>10.6f}   {cf_mae:>10.6f}   {'(baseline)'}")
print(f"{'Hybrid CF+Content':<25} {rmse:>10.6f}   {mae:>10.6f}   {(cf_rmse-rmse)/cf_rmse*100:>+6.2f}% RMSE")

improvement = (cf_rmse - rmse) / cf_rmse * 100
if improvement > 0:
    print(f"\n✓ Hybrid system improves RMSE by {improvement:.2f}%")
else:
    print(f"\n✗ Hybrid system increases RMSE by {abs(improvement):.2f}%")

## 9. Visualize Results

In [ ]:
# Create comparison visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: RMSE and MAE comparison
algorithms = ['Pure CF', 'Hybrid']
rmse_values = [cf_rmse, rmse]
mae_values = [cf_mae, mae]

x = np.arange(len(algorithms))
width = 0.35

axes[0].bar(x - width/2, rmse_values, width, label='RMSE', color='steelblue', alpha=0.8)
axes[0].bar(x + width/2, mae_values, width, label='MAE', color='coral', alpha=0.8)
axes[0].set_ylabel('Error', fontsize=12)
axes[0].set_title('Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(algorithms)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Add value labels on bars
for i, (r, m) in enumerate(zip(rmse_values, mae_values)):
    axes[0].text(i - width/2, r + 0.01, f'{r:.4f}', ha='center', va='bottom', fontsize=10)
    axes[0].text(i + width/2, m + 0.01, f'{m:.4f}', ha='center', va='bottom', fontsize=10)

# Plot 2: Breakdown by case type
case_counts = results_df['case'].value_counts()
case_colors = {'warm': 'green', 'partial_cold': 'orange', 'double_cold': 'red'}
colors = [case_colors.get(case, 'gray') for case in case_counts.index]

axes[1].bar(range(len(case_counts)), case_counts.values, color=colors, alpha=0.7)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Prediction Distribution by Case Type', fontsize=14, fontweight='bold')
axes[1].set_xticks(range(len(case_counts)))
axes[1].set_xticklabels([c.replace('_', ' ').title() for c in case_counts.index], rotation=15)
axes[1].grid(axis='y', alpha=0.3)

# Add percentage labels
for i, (count, case) in enumerate(zip(case_counts.values, case_counts.index)):
    pct = count / len(results_df) * 100
    axes[1].text(i, count + len(results_df)*0.01, f'{count:,}\n({pct:.1f}%)', 
                ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('hybrid_comparison.png', dpi=300, bbox_inches='tight')
print("\nVisualization saved as 'hybrid_comparison.png'")
plt.show()

## 10. Save Results

In [ ]:
# Save detailed results to CSV
comparison_results = pd.DataFrame({
    'Algorithm': ['Pure Item-Based CF', 'Hybrid CF+Content'],
    'RMSE': [cf_rmse, rmse],
    'MAE': [cf_mae, mae],
    'Test_Samples': [len(test_sample), len(test_sample)],
    'Notes': [
        f'Fallback: {cf_fallback_count} ({cf_fallback_count/len(test_sample)*100:.2f}%)',
        f'Warm: {len(results_df[results_df["case"]=="warm"])}, Partial: {len(results_df[results_df["case"]=="partial_cold"])}, Double: {len(results_df[results_df["case"]=="double_cold"])}'
    ]
})

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
filename = f'hybrid_comparison_{timestamp}.csv'
comparison_results.to_csv(filename, index=False)
print(f"Results saved to {filename}")

# Save detailed breakdown
case_breakdown = []
for case_type in ['warm', 'partial_cold', 'double_cold']:
    case_df = results_df[results_df['case'] == case_type]
    if len(case_df) > 0:
        case_rmse = np.sqrt(mean_squared_error(case_df['actual'], case_df['predicted']))
        case_mae = mean_absolute_error(case_df['actual'], case_df['predicted'])
        case_breakdown.append({
            'Case': case_type,
            'Count': len(case_df),
            'Percentage': len(case_df) / len(results_df) * 100,
            'RMSE': case_rmse,
            'MAE': case_mae
        })

breakdown_df = pd.DataFrame(case_breakdown)
breakdown_filename = f'hybrid_case_breakdown_{timestamp}.csv'
breakdown_df.to_csv(breakdown_filename, index=False)
print(f"Case breakdown saved to {breakdown_filename}")

print(f"\n{'='*80}")
print("HYBRID SYSTEM IMPLEMENTATION COMPLETE")
print(f"{'='*80}")

## Summary

This hybrid recommender system successfully combines:
1. **Item-Based Collaborative Filtering** for users/movies with sufficient interaction history
2. **Content-Based Filtering** using genre information for cold-start scenarios
3. **Adaptive weighting** that adjusts the balance based on data availability

The hybrid approach demonstrates the ability to handle various scenarios:
- **Warm-start**: Leverages collaborative patterns (70% weight)
- **Partial cold-start**: Balances CF and content (30% CF, 70% content)
- **Double cold-start**: Falls back to content-based recommendations

This addresses one of the key limitations identified in pure collaborative filtering approaches and provides a more robust solution for production deployment.